## Prepare Env

In [2]:
!python -m venv venv
!source venv/bin/activate

In [3]:
%pip install boto3 pyspark delta-spark python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 758.9 kB/s eta 0:00:0000:0100:11
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 269.5 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 238.8 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 554.6 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 804.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 kB 741.7 kB/s eta 0:00:00a 0:00:01
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425345 sha256=2e5f5bf5d46b0c9bb54a05b8100bd25893f406bfe494b9d59a2cf802fad0df7d
  Stored in directory: /Users/quangtu/Library/Caches/pip/wheels/84/40/20/65eefe766118e0a8f8e385cc3ed6e9eb7241c7e51cfc04c51a
Successfully built pyspark
Note: you may 

In [4]:
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os
from dotenv import load_dotenv

In [5]:
load_dotenv()

False

In [5]:
# Define S3 storage
obj_storage_access_key = os.getenv('OBJ_STORAGE_ACCESS_KEY', 'demo-access-key')
obj_storage_secret_key = os.getenv('OBJ_STORAGE_SECRET_KEY', 'demo-secret-key')
obj_storage_endpoint = os.getenv('OBJ_STORAGE_ENDPOINT', 'http://localhost:9000')

## Ingestion
### 1. Layer files to layer bronze
Write files which are in layer files to delta table in layer bronze

In [6]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("CsvToDelta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

24/01/17 15:54:56 WARN Utils: Your hostname, Lukes-Macbook.local resolves to a loopback address: 127.0.0.1; using 192.168.1.18 instead (on interface en0)
24/01/17 15:54:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/quangtu/Documents/Codes/icttm/adam_datalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/quangtu/.ivy2/cache
The jars for the packages stored in: /Users/quangtu/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7c472d36-ef51-4d1c-9cdf-cdecb82ac8a7;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.1 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.901 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 145ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 from central in [default]
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.h

In [7]:
file_path = "s3a://warehouse/files/gleif.file/data.csv"
delta_table_path = "s3a://warehouse/bronze/gleif_entities.delta"

In [10]:
# Read file into a DataFrame
df = spark.read.csv(file_path, header=True, sep=",")

24/01/17 15:55:16 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [11]:
df.show()

24/01/17 15:55:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+--------------------+------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+---------------------------------------------------------------------+-----------------------------------------------------------------------------+--------------------------------------------------------------------------+--------------

In [12]:
# Write DataFrame to Delta table
df.write.format("delta").mode("overwrite").save(delta_table_path)

# Stop the Spark session
spark.stop()

24/01/17 15:55:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:41 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:47 WARN MemoryManager: Total allocation exceeds 95.00% 

# Processing

Processing these step before writing data to layer silver
1. Transform to standard schema of layer silver
2. Unique each record
3. Add fields
4. Map entities
5. Upsert

Read delta table and discovery data

In [13]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("CsvToDelta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

24/01/17 15:57:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [14]:
delta_table_path = "s3a://warehouse/bronze/gleif_entities.delta"

In [15]:
# Read Delta table
df = spark.read.format("delta").load(delta_table_path)

In [16]:
df.show()

+--------------------+--------------------+------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+---------------------------------------------------------------------+-----------------------------------------------------------------------------+--------------------------------------------------------------------------+--------------